In [12]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer
from datasets import load_dataset
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments
from transformers import Trainer
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, precision_score, recall_score,accuracy_score
from google.colab import drive

save_path_bbu = "/content/drive/MyDrive/model/bbu_model"
save_path_dbbu = "/content/drive/MyDrive/model/dbbu_model"
drive.mount('/content/drive')
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuda


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
bert_base_uncased = "bert-base-uncased"
distilbert_base_uncased = "distilbert-base-uncased"
tokenizer_bbu = AutoTokenizer.from_pretrained(bert_base_uncased)
tokenizer_dbbu = AutoTokenizer.from_pretrained(distilbert_base_uncased)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
dataset = load_dataset("SetFit/ag_news")
train_perc = int(dataset.num_rows['train']*0.12)
test_perc = int(dataset.num_rows['test']*0.1)
train_set_bef = dataset['train'][:train_perc]
test_set = dataset['test'][:test_perc]
train_perc = int(len(train_set_bef['text'])*0.9)
train_set = {}
val_set = {}
for k,v in train_set_bef.items():
  train_set[k] = train_set_bef[k][:train_perc]
for k,v in train_set_bef.items():
  val_set[k] = train_set_bef[k][train_perc:]
print("Train split : ",len(train_set['label']))
print("Test split  : ",len(test_set['label']))
print("Val split   : ",len(val_set['label']))

Train split :  12960
Test split  :  760
Val split   :  1440


#PREPROCESSING TEXT

In [16]:
def process_text(text):
  text_lower = text.lower()
  text_rspecial = re.sub(r'[^a-zA-Z0-9 ]', '', text_lower)
  stop_words = set(stopwords.words('english'))
  tokens = word_tokenize(text_rspecial)
  toke_bef = len(tokens)
  filtered_tokens = [word for word in tokens if word not in stop_words]
  toke_aft = len(filtered_tokens)
  clean_text = ' '.join(filtered_tokens)
  return clean_text,toke_bef,toke_aft

# ADDING NEW COLOUMN TO DATASET AND COUNTING TOKEN DISTRIBUTION BEFORE AND AFTER CLEANING

In [17]:
train_set['clean_text'] = []
test_set['clean_text'] = []
val_set['clean_text'] = []
def add_clean_text(t_set):
  c_t = []
  tok_bef = 0;
  tok_aft = 0;
  for t in t_set['text']:
    clean,b,a = process_text(t)
    tok_bef += b
    tok_aft += a
    c_t.append(clean)
  t_set['clean_text'] = c_t
  return tok_bef,tok_aft
tr_b,tr_a = add_clean_text(train_set)
ts_b,ts_a = add_clean_text(test_set)
vl_b,vl_a = add_clean_text(val_set)
print("token distribution before cleaning : ",tr_b+ts_b+vl_b)
print("token distribution after cleaning  : ",tr_a+ts_a+vl_a)

token distribution before cleaning :  577029
token distribution after cleaning  :  397082


#SAVING DATASETS ALONG WITH CLEANED TEXT

In [32]:
import pickle

with open("/content/drive/MyDrive/dataset/Train_set.pkl", "wb") as f:
    pickle.dump(train_set, f)
with open("/content/drive/MyDrive/dataset/Test_set.pkl", "wb") as f:
    pickle.dump(test_set, f)

#PREPARING DATASETS TO TRAIN

In [10]:
class CustomDataset(Dataset):
  def __init__(self,tokenizer,dataset):
    self.tokenizer = tokenizer
    self.dataset = dataset
  def __getitem__(self, index):
    input = self.tokenizer(self.dataset['clean_text'][index],truncation=True,return_tensors = "pt",max_len = 256,padding='max_length')
    item = {
        'input_ids' : input['input_ids'].squeeze(0),
        'attention_mask' : input['attention_mask'].squeeze(0),
        'labels' : torch.tensor(self.dataset['label'][index],dtype=torch.long)
    }
    return item
  def __len__(self):
    return len(self.dataset['label'])

val_train = {}
val_test  = {}
for k,v in val_set.items():
  val_train[k] = val_set[k][:int(len(val_set['label'])*0.8)]
for k,v in val_set.items():
  val_test[k] = val_set[k][int(len(val_set['label'])*0.8):]
print(len(val_test['label']),len(val_train['label']))
train_ds_bbu = CustomDataset(tokenizer_bbu,train_set)
test_ds_bbu  = CustomDataset(tokenizer_bbu,test_set)
val_train_ds_bbu = CustomDataset(tokenizer_bbu,val_train)
val_test_ds_bbu  = CustomDataset(tokenizer_bbu,val_test)

train_ds_dbbu = CustomDataset(tokenizer_dbbu,train_set)
test_ds_dbbu  = CustomDataset(tokenizer_dbbu,test_set)
val_train_ds_dbbu = CustomDataset(tokenizer_dbbu,val_train)
val_test_ds_dbbu  = CustomDataset(tokenizer_dbbu,val_test)

288 1152


#HYPER PARAMETER TUNING

In [13]:
def fetch_best_hps(model_name,epochs,train_ds,val_ds,learning_rates=[1e-5,2e-5,1e-4],batch_sizes=[4,8,16]):
  best_params = {'learning_rate' : None, 'batch_size' : None, 'accuracy' : -1}
  for lr in learning_rates:
    for bs in batch_sizes:
      model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=4).to(device)
      training_args = TrainingArguments(num_train_epochs=epochs,per_device_train_batch_size=bs,learning_rate=lr,eval_strategy="epoch",logging_strategy="epoch")
      trainer = Trainer(model=model,train_dataset=train_ds,eval_dataset = val_ds,args=training_args)
      trainer.train()
      val_loader = DataLoader(val_ds,batch_size=bs)
      count = 0
      hit_count = 0
      for batch in val_loader:
        batch['input_ids'] = batch['input_ids'].to(device)
        batch['attention_mask'] = batch['attention_mask'].to(device)
        batch['labels'] = batch['labels'].to(device)
        out = model(**batch)
        preds = torch.argmax(out['logits'],dim = 1)
        count += bs
        for i,j in zip(preds,batch['labels']):
          if i==j:
            hit_count += 1
      accuracy = hit_count/count
      print(f"Learning rate : {lr} , batch size : {bs} , Accuracy : {accuracy}")
      if accuracy>best_params['accuracy']:
        best_params['learning_rate'] = lr
        best_params['batch_size'] = bs
        best_params['accuracy'] = accuracy
  return best_params

#LOADING BEST HYPER PARAMETERS

In [14]:
best_params_bbu = fetch_best_hps(bert_base_uncased,2,val_train_ds_bbu,val_test_ds_bbu)
print(best_params_bbu)
best_params_dbbu = fetch_best_hps(distilbert_base_uncased,2,val_train_ds_dbbu,val_test_ds_dbbu)
print(best_params_bbu)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.690812,0.544420
2,0.304936,0.464772


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.528892,0.490701
2,0.211987,0.517479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Learning rate : 0.0001 , batch size : 16 , Accuracy : 0.8784722222222222
{'learning_rate': 0.0001, 'batch_size': 8, 'accuracy': 0.8923611111111112}


#CODE FOR TRAINING AND COMPUTING APPROPRIATE METRICS OF THE TRAINED MODEL

In [15]:
def trainer_code(model_name,best_params,train_ds,test_ds,epochs):
  model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=4).to(device)
  bs = best_params['batch_size']
  lr = best_params['learning_rate']
  training_args = TrainingArguments(num_train_epochs=epochs,per_device_train_batch_size=bs,learning_rate=lr,eval_strategy="epoch",logging_strategy="epoch")
  trainer = Trainer(model=model,train_dataset=train_ds,eval_dataset = test_ds,args=training_args)
  trainer.train()
  val_loader = DataLoader(test_ds,batch_size=bs)
  count = 0
  hit_count = 0
  y_pred = []
  y_true = []
  for batch in val_loader:
    batch['input_ids'] = batch['input_ids'].to(device)
    batch['attention_mask'] = batch['attention_mask'].to(device)
    batch['labels'] = batch['labels'].to(device)
    y_true.extend(batch['labels'].tolist())
    out = model(**batch)
    preds = torch.argmax(out['logits'],dim = 1)
    y_pred.extend(preds.tolist())
  cm = confusion_matrix(y_true, y_pred)
  print("Confusion Matrix:\n", cm)

  precision = precision_score(y_true, y_pred, average=None)
  recall = recall_score(y_true, y_pred, average=None)
  accuracy = accuracy_score(y_true,y_pred)

  print("Accuracy : ",accuracy)
  print("Precision per class:", precision)
  print("Recall per class:", recall)

  print("Macro Precision:", precision_score(y_true, y_pred, average='macro'))
  print("Macro Recall:", recall_score(y_true, y_pred, average='macro'))
  print("Micro Precision:", precision_score(y_true, y_pred, average='micro'))
  print("Micro Recall:", recall_score(y_true, y_pred, average='micro'))
  return model

#TRAINING THE MODELS

In [16]:
print("Bert Base Uncased : ")
model_bbu = trainer_code(bert_base_uncased,best_params_bbu,train_ds_bbu,test_ds_bbu,2)
print("------------------------------------------------------------------------------")
print("distilbert base uncased")
model_dbbu = trainer_code(distilbert_base_uncased,best_params_dbbu,train_ds_dbbu,test_ds_dbbu,2)

Bert Base Uncased : 


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.521038,0.453339
2,0.354836,0.370560


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Confusion Matrix:
 [[179   6  10   2]
 [  4 193   0   2]
 [ 10   0 134  14]
 [  2   1  11 192]]
Accuracy :  0.9184210526315789
Precision per class: [0.91794872 0.965      0.86451613 0.91428571]
Recall per class: [0.90862944 0.96984925 0.84810127 0.93203883]
Macro Precision: 0.9154376403166725
Macro Recall: 0.9146546971574406
Micro Precision: 0.9184210526315789
Micro Recall: 0.9184210526315789


#SAVING THE BEST MODELS

In [17]:
model_bbu.save_pretrained(save_path_bbu)
tokenizer_bbu.save_pretrained(save_path_bbu)
model_dbbu.save_pretrained(save_path_dbbu)
tokenizer_dbbu.save_pretrained(save_path_dbbu)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/model/dbbu_model/tokenizer_config.json',
 '/content/drive/MyDrive/model/dbbu_model/tokenizer.json')

In [3]:
from google.colab import drive
from transformers import AutoModelForSequenceClassification,AutoTokenizer
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
save_path_bbu = "/content/drive/MyDrive/model/bbu_model/"
save_path_dbbu = "/content/drive/MyDrive/model/dbbu_model/"

#LOADING THE BEST MODELS FROM GOOGLE DRIVE

In [5]:
model_bbu = AutoModelForSequenceClassification.from_pretrained(save_path_bbu)
tokenizer_bbu = AutoTokenizer.from_pretrained(save_path_bbu)
model_dbbu = AutoModelForSequenceClassification.from_pretrained(save_path_dbbu)
tokenizer_dbbu = AutoTokenizer.from_pretrained(save_path_dbbu)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

In [6]:
import pickle
with open("/content/drive/MyDrive/dataset/Train_set.pkl", "rb") as f:
    train_ds = pickle.load(f)
with open("/content/drive/MyDrive/dataset/Test_set.pkl", "rb") as f:
    test_ds = pickle.load(f)

In [7]:
train_ds['text_vector'] = []
test_ds['text_vector']  = []

#Embedding generation

In [8]:
import torch
for t in train_ds['clean_text']:
  inputs = tokenizer_bbu(t,return_tensors="pt",truncation=True,max_length=256)
  outputs = model_bbu(**inputs,output_hidden_states=True)
  train_ds['text_vector'].append(torch.mean(outputs.hidden_states[-1],dim=1).squeeze(0).tolist())

In [25]:
for t in test_ds['clean_text']:
  inputs = tokenizer_dbbu(t,return_tensors="pt",truncation=True,max_length=256)
  outputs = model_dbbu(**inputs,output_hidden_states=True)
  test_ds['text_vector'].append(torch.mean(outputs.hidden_states[-1],dim=1).squeeze(0).tolist())

#SAVING THE EMBEDDINGS TO DRIVE

In [ ]:
import pickle

with open("/content/drive/MyDrive/dataset/Train_set_we.pkl", "wb") as f:
    pickle.dump(train_ds, f)
with open("/content/drive/MyDrive/dataset/Test_set_we.pkl", "wb") as f:
    pickle.dump(test_ds, f)

#SEMANTIC SEARCH BASED ON USER QUERY USING STORES EMBEDDINGS

In [49]:
import numpy as np
from numpy.linalg import norm
def semantic_search(query,ds,model,tokenizer,top_k=5):
  print("QUERY : ",query)
  print("-------------------------------------------------------------------------------------")
  similarity_scores = []
  inputs = tokenizer(query,return_tensors="pt",truncation=True,max_length=256)
  outputs = model(**inputs,output_hidden_states=True)
  query_vector = torch.mean(outputs.hidden_states[-1],dim=1).squeeze(0).tolist()
  query_vector_np = np.array(query_vector)
  for v in ds['text_vector']:
    v_np = np.array(v)
    similarity_scores.append(np.dot(query_vector_np, v_np) / (norm(query_vector_np) * norm(v_np)))
  sim_scores_tensor = torch.tensor(similarity_scores)
  topk_articles = torch.topk(sim_scores_tensor,top_k,sorted = False)
  indices = topk_articles.indices.tolist()
  for i in indices:
    print("Article : ", ds['text'][i])
    print("category : ", ds['label_text'][i])
    print("score : ", similarity_scores[i])
    print("")


#SEMANTIC SEARCH USING BERT BASE UNCASED

In [52]:
semantic_search("Global economic inflation and market crash",train_ds,model_bbu,tokenizer_bbu)
semantic_search("kabaddi is a game which is famous in india, people get excited watching it",train_ds,model_bbu,tokenizer_bbu)
semantic_search("attention is all you need is a paper which revolutionized ai through out the world",train_ds,model_bbu,tokenizer_bbu)

QUERY :  Global economic inflation and market crash
-------------------------------------------------------------------------------------
Article :  Stocks End Up; Techs Gain on Light Volume (Reuters) Reuters - U.S. blue-chips closed at six-week\highs on Friday, helped by reassuring consumer confidence data\and a bounce in technology stocks, but volume was extremely\thin before next week's Republican convention in New York City.
category :  Business
score :  0.9982500224581908

Article :  Stocks off slightly despite oil news A brokerage firm #39;s negative outlook for the semiconductor industry pushed down tech shares yesterday, while the broader market was little changed as a jump in jobless claims offset investors #39; relief over declining oil prices.
category :  Business
score :  0.9979310316808969

Article :  Synovis shares tumble as profit, revenue fall CHICAGO, Aug 18 (Reuters) - Shares of Synovis Life Technologies Inc.(SYNO.O: Quote, Profile, Research) tumbled 13 percent on Wed

#SEMANTIC SEARCH USING DISTILBERT BASE UNCASED

In [53]:
semantic_search("Global economic inflation and market crash",train_ds,model_dbbu,tokenizer_dbbu)
semantic_search("kabaddi is a game which is famous in india, people get excited watching it",train_ds,model_dbbu,tokenizer_dbbu)
semantic_search("attention is all you need is a paper which revolutionized ai through out the world",train_ds,model_dbbu,tokenizer_dbbu)

QUERY :  Global economic inflation and market crash
-------------------------------------------------------------------------------------
Article :  Iraq Keeps South Oil Pipeline Shut  BAGHDAD (Reuters) - Authorities kept a main oil pipeline in  southern Iraq shut on Sunday rather than risk it being  attacked, restricting the country's exports to half normal  levels, a South Oil Official said.
category :  Business
score :  0.3078791262755016

Article :  Taiwan's Economy Expands at Fast Pace (AP) AP - Taiwan's economy expanded at its fastest pace in four years during the second quarter as the global economy recovered, boosting foreign trade and increasing manufacturing output, the government said Friday.
category :  Business
score :  0.30723022465234534

Article :  South Korea in growth spurt South Korea's economy grew faster than expected in the second quarter, but growth is still held back by poor domestic demand.
category :  Business
score :  0.30563889144376016

Article :  Iraq Halt